## Sample Generation for Stable Crop

In [ ]:
import ee
ee.Initialize(project="gee231")
import geemap

In [ ]:
FILTERED   = ee.FeatureCollection("projects/ee-joshisur231/assets/agriculture_abandonment_nepal/filtered_points/stable_ag_points_filtered")
ALREADY    = geemap.csv_to_ee(r"outputs\training_samples_for_RF\stable_crop_samples_manually_validated_withGeo.csv", longitude="x", latitude="y")  # the crop points you already have
GEOREGION  = ee.Image("projects/ee-joshisur231/assets/pa_effectiveness/geoReg_nepal") 
HAS_GEOREGION = False          # set True if FILTERED already has a 'geoRegion' property (then GEOREGION unused)

MIN_DIST      = 1000           
N_PER_REGION  = {1: 60, 2: 300}   
SEED          = 45

if HAS_GEOREGION:
    cand = FILTERED
else:
    cand = (GEOREGION.rename('geoRegion').reduceRegions(
                collection=FILTERED, reducer=ee.Reducer.first(), scale=100)
                .filter(ee.Filter.notNull(['first']))
            .map(lambda f: f.set('geoRegion', ee.Number(f.get('first')).toInt())))

cand = cand.filter(ee.Filter.notNull(['geoRegion']))
cand = cand.filter(ee.Filter.inList('geoRegion', [1, 2]))

old = ALREADY.map(lambda f: ee.Feature(f.geometry()).set('rnd', 2).set('isnew', 0))
new = cand.randomColumn('rnd', SEED).map(lambda f: f.set('isnew', 1))
merged = old.merge(new)

dist_filter = ee.Filter.withinDistance(distance=MIN_DIST, leftField='.geo',
                                        rightField='.geo', maxError=10)
joined = ee.Join.saveAll(matchesKey='nbrs').apply(merged, merged, dist_filter)

def flag_keep(f):
    nbrs = ee.FeatureCollection(ee.List(f.get('nbrs')))
    return f.set('keep', ee.Number(f.get('rnd')).gte(nbrs.aggregate_max('rnd')))

kept_new = (ee.FeatureCollection(joined).map(flag_keep)
            .filter(ee.Filter.eq('keep', 1))
            .filter(ee.Filter.eq('isnew', 1)))

print('survivors per region (region 1 / region 2):')
print(kept_new.filter(ee.Filter.eq('geoRegion', 1)).size().getInfo(),
      kept_new.filter(ee.Filter.eq('geoRegion', 2)).size().getInfo())

def take(region, n):
    return (kept_new.filter(ee.Filter.eq('geoRegion', region))
            .randomColumn('r2', SEED + region).limit(n, 'r2'))

final = take(1, N_PER_REGION[1]).merge(take(2, N_PER_REGION[2]))

final = final.map(lambda f: f.set('x', f.geometry().coordinates().get(0))
                             .set('y', f.geometry().coordinates().get(1)))
print('final selected:', final.size)

survivors per region (region 1 / region 2):
649 2791
final selected: <bound method Collection.size of <ee.featurecollection.FeatureCollection object at 0x000002730CDA9BB0>>


In [34]:
Map = geemap.Map()
Map.addLayer(final.style(color="red"),{}, "final")
Map.addLayer(ALREADY.style(color="blue"), {}, "already" )
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [ ]:
ee.batch.Export.table.toDrive(
    collection=final, description='stable_ag_extra_r1r2', fileFormat='CSV',
    selectors=['geoRegion', 'x', 'y', "isnew"]).start()

## Sample Generation for Stable Non-Crop

In [1]:
import ee
ee.Initialize(project="ee-joshisur231")
import geemap
import geopandas as gpd
Map = geemap.Map()

In [2]:
ALREADY = geemap.shp_to_ee(r"outputs\scratch\stable_nonCrop_manually_validated.shp")

In [33]:
# N_PER_REGION =     {'1_1': 7,
#                     # '1_4':,
#                     '1_5': 10,
#                     '1_6': 8,
#                     '1_9': 7,
#                     # '1_10': ,
#                     # '1_11': ,
#                     '2_1': 18,
#                     '2_4': 85,
#                     '2_5': 25,
#                     '2_6': 55,
#                     '2_9': 15,
#                     '2_10': 50,
#                     '2_11': 50,
#                     '3_1': 23,
#                     '3_4': 58,
#                     '3_5': 29,
#                     '3_6': 68,
#                     '3_10': 30,
#                     '3_11': 28,
#                     }
N_PER_REGION =     {'1_1': 7,
                    # '1_4':,
                    '1_5': 10,
                    '1_6': 8,
                    '1_9': 73,
                    # '1_10': ,
                    # '1_11': ,
                    '2_1': 18,
                    '2_4': 141,
                    '2_5': 81,
                    '2_6': 55,
                    '2_9': 15,
                    '2_10': 106,
                    '2_11': 106,
                    '3_1': 23,
                    '3_4': 58,
                    '3_5': 95,
                    '3_6': 68,
                    '3_10': 30,
                    '3_11': 28,
                    }

In [34]:
total = 0
total_1 = 0
total_2 = 0
total_3 = 0
for key, value in N_PER_REGION.items():
    total+= value
    if int(key.split("_")[0]) == 1:
        total_1 += value
    elif int(key.split("_")[0]) == 2:
        total_2 += value
    elif int(key.split("_")[0]) == 3:
        total_3 += value
    else:
        raise ValueError()
print(f"total = {total} \n1 = {109+total_1} \n2 = {183+total_2} \n3 = {165+total_3}")

total = 922 
1 = 207 
2 = 705 
3 = 467


In [44]:
GEOREGION  = ee.Image("projects/ee-joshisur231/assets/pa_effectiveness/geoReg_nepal") 
frtc_lc_ic = ee.ImageCollection("projects/ee-joshisur231/assets/landcover_frtc_2000-2022_nepal").mode() #.filter(ee.Filter.eq("system:index", "lc2022")).first()

FILTERED = ee.FeatureCollection("projects/ee-joshisur231/assets/agriculture_abandonment_nepal/crop_nonCrop_training_samples_glad").filter(ee.Filter.eq("stable_crop", 0))
FILTERED = GEOREGION.reduceRegions(collection=FILTERED, reducer=ee.Reducer.first(), scale=100).filter(ee.Filter.notNull(['first'])).map(lambda f: f.set('geoRegion', ee.Number(f.get('first')).toInt()))
FILTERED = frtc_lc_ic.reduceRegions(collection=FILTERED, reducer=ee.Reducer.first(), scale=30).filter(ee.Filter.notNull(['first'])).map(lambda f: f.set('frtc_lc', ee.Number(f.get('first')).toInt()))

FILTERED = FILTERED.map(lambda f: f.set("geoRegion_lc", ee.String(f.get("geoRegion")).cat(ee.String("_")).cat(ee.String(f.get("frtc_lc")))))

ALREADY = GEOREGION.reduceRegions(collection=ALREADY, reducer=ee.Reducer.first(), scale=100).filter(ee.Filter.notNull(['first'])).map(lambda f: f.set('geoRegion', ee.Number(f.get('first')).toInt()))
ALREADY = frtc_lc_ic.reduceRegions(collection=ALREADY, reducer=ee.Reducer.first(), scale=30).filter(ee.Filter.notNull(['first'])).map(lambda f: f.set('frtc_lc', ee.Number(f.get('first')).toInt()))

ALREADY = ALREADY.map(lambda f: f.set("geoRegion_lc", ee.String(f.get("geoRegion")).cat(ee.String("_")).cat(ee.String(f.get("frtc_lc")))))

In [37]:
GEOREGION  = ee.Image("projects/ee-joshisur231/assets/pa_effectiveness/geoReg_nepal") 
MIN_DIST      = 1000           
SEED          = 45

cand = FILTERED
cand = cand.filter(ee.Filter.notNull(['geoRegion_lc']))

old = ALREADY.map(lambda f: ee.Feature(f.geometry()).set('rnd', 2).set('isnew', 0))
new = cand.randomColumn('rnd', SEED).map(lambda f: f.set('isnew', 1))
merged = old.merge(new)

dist_filter = ee.Filter.withinDistance(distance=MIN_DIST, leftField='.geo',
                                        rightField='.geo', maxError=10)
joined = ee.Join.saveAll(matchesKey='nbrs').apply(merged, merged, dist_filter)

def flag_keep(f):
    nbrs = ee.FeatureCollection(ee.List(f.get('nbrs')))
    return f.set('keep', ee.Number(f.get('rnd')).gte(nbrs.aggregate_max('rnd')))

kept_new = (ee.FeatureCollection(joined).map(flag_keep)
            .filter(ee.Filter.eq('keep', 1))
            .filter(ee.Filter.eq('isnew', 1)))

print('survivors per region (region 1 / region 2):')
print(kept_new.filter(ee.Filter.eq('geoRegion', 1)).size().getInfo(),
      kept_new.filter(ee.Filter.eq('geoRegion', 2)).size().getInfo(),
      kept_new.filter(ee.Filter.eq('geoRegion', 3)).size().getInfo())

def take(region_str, n):
    # Convert string like '1_5' to an integer 15 for the seed
    local_seed = SEED + int(region_str.replace('_', ''))
    return (kept_new.filter(ee.Filter.eq('geoRegion_lc', region_str))
            .randomColumn('r2', local_seed).limit(n, 'r2'))

final_list = []
for reg_code, count in N_PER_REGION.items():
    final_list.append(take(reg_code, count))

final = ee.FeatureCollection(final_list).flatten()

final = final.map(lambda f: f.set('x', f.geometry().coordinates().get(0))
                             .set('y', f.geometry().coordinates().get(1)))

print('Final selected:', final.size().getInfo())

survivors per region (region 1 / region 2):
677 484 165
Final selected: 569


In [47]:
merged = ALREADY.merge(final).select(["geoRegion_lc", "x", "y"])

In [70]:
merged.aggregate_histogram("geoRegion_lc")

In [49]:
merged.size()

In [62]:
merged_list = merged.toList(merged.size())
index_list = ee.List.sequence(0, merged.size().subtract(1))

def add_id(index):
  id = ee.Number(index).int().add(1)
  feat = ee.Feature(merged_list.get(index))
  return feat.set("id", id)
merged = ee.FeatureCollection(index_list.map(add_id))

In [69]:
merged.filter(ee.Filter.eq("geoRegion_lc", "2_7"))

In [65]:
ee.batch.Export.table.toDrive(
    collection=merged, description='stable_noncrop_combined_CSV', fileFormat='CSV').start()

In [20]:
total_difference = 0

for reg_code, requested_n in N_PER_REGION.items():
    available = kept_new.filter(ee.Filter.eq('geoRegion_lc', reg_code)).size().getInfo()
    difference = requested_n-available
    if available < requested_n:
        print(f"Class {reg_code}: Requested {requested_n}, Available {available}, Difference {difference}")
        total_difference += difference
    

Class 1_1: Requested 7, Available 2, Difference 5
Class 1_5: Requested 10, Available 2, Difference 8
Class 1_6: Requested 8, Available 0, Difference 8
Class 2_1: Requested 18, Available 3, Difference 15
Class 2_5: Requested 65, Available 38, Difference 27
Class 2_6: Requested 55, Available 7, Difference 48
Class 2_9: Requested 15, Available 7, Difference 8
Class 2_10: Requested 90, Available 86, Difference 4
Class 2_11: Requested 90, Available 56, Difference 34
Class 3_1: Requested 23, Available 4, Difference 19
Class 3_4: Requested 58, Available 26, Difference 32
Class 3_6: Requested 68, Available 5, Difference 63
Class 3_10: Requested 30, Available 24, Difference 6
Class 3_11: Requested 28, Available 0, Difference 28


In [21]:
total_difference

305

In [14]:
240/6

40.0

In [15]:
for reg_code, requested_n in N_PER_REGION.items():
    available = kept_new.filter(ee.Filter.eq('geoRegion_lc', reg_code)).size().getInfo()
    if available >= requested_n:
        print(f"Class {reg_code}: Requested {requested_n}, Available {available}, Added {requested_n + 40}")

Class 1_9: Requested 7, Available 213, Added 47
Class 2_4: Requested 85, Available 287, Added 125
Class 2_5: Requested 25, Available 38, Added 65
Class 2_10: Requested 50, Available 86, Added 90
Class 2_11: Requested 50, Available 56, Added 90
Class 3_5: Requested 29, Available 106, Added 69


In [ ]:
N_PER_REGION =     {'1_1': 7,
                    # '1_4':,
                    '1_5': 10,
                    '1_6': 8,
                    '1_9': 63,
                    # '1_10': ,
                    # '1_11': ,
                    '2_1': 18,
                    '2_4': 141,
                    '2_5': 81,
                    '2_6': 55,
                    '2_9': 15,
                    '2_10': 106,
                    '2_11': 106,
                    '3_1': 23,
                    '3_4': 58,
                    '3_5': 85,
                    '3_6': 68,
                    '3_10': 30,
                    '3_11': 28,
                    }

In [27]:
69+16

85